# EdgeFace-S → Indian-face fine-tune (ArcFace) → TFLite for Datalake Face Auth

**Goal.** Take the SOTA compact face-recognition backbone **EdgeFace-S** (IJCB-2023
efficient-FR winner, 1.77 M params, 512-d, 99.73% LFW) and adapt it to **Indian
faces**, then export a **TFLite** model that drops straight into the offline
React Native app (`app/assets/models/edgeface_s.tflite`).

**Why this is the winning "extra".** The stock backbone is demographic-general.
Here we (1) **fine-tune with ArcFace** on an identity-labelled Indian dataset
(IMFDB), and (2) **quantify demographic robustness** on IndicFairFace by measuring
inter-state variance in similarity **before vs after** — a concrete, honest number
you can put on a slide, not a vague "we support Indian faces" claim.

**Constraints kept:** output is 112×112×3 in, 512-d out, mean/std 0.5 — identical
to `app/src/config.ts` `RECOGNITION_MODELS.edgeface_s`, so no app change beyond
dropping in the file and flipping `ACTIVE_RECOGNITION`.

**Hardware:** any CUDA GPU (your 40 GB is far more than enough — this is a small
model + small dataset; expect minutes/epoch).

---
### Pipeline
1. Environment
2. Load EdgeFace-S backbone (+ sanity check I/O)
3. Align faces (MTCNN) → 112×112 ImageFolder
4. Data loaders (ArcFace normalisation)
5. ArcFace margin head
6. Assemble model + optimiser (freeze → unfreeze schedule)
7. Verification-accuracy evaluation
8. Train
9. IndicFairFace bias evaluation (before vs after)
10. Export → ONNX → **float16 TFLite** (+ optional INT8)
11. Verify the TFLite I/O and drop into the app


## 1 · Environment

Run once. Pins the pieces that matter; everything is open-source.

In [ ]:
%pip install -q \
    torch torchvision timm \
    facenet-pytorch \
    onnx onnxruntime onnx2tf \
    "tensorflow==2.17.*" \
    scikit-learn matplotlib opencv-python-headless tqdm
# onnx2tf is the reliable ONNX(NCHW) -> TFLite(NHWC) path in 2026.
# (Alternative: Google's `ai-edge-torch` for direct PyTorch->TFLite.)

In [ ]:
import os, math, random, glob, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import ImageFolder

CFG = {
    # Raw dataset: one folder per identity, unaligned images.
    #   data/indian_faces_raw/<identity_id>/<img>.jpg
    "data_root_raw": "data/indian_faces_raw",
    # Produced by the alignment step (step 3):
    "data_root":     "data/indian_faces_aligned",
    # Optional held-out identities for an HONEST verification metric:
    "val_root":      "data/indian_faces_val_aligned",
    # Optional IndicFairFace, aligned, grouped by state for the bias metric:
    #   data/indicfairface_aligned/<state>/<identity>/<img>.jpg
    "bias_root":     "data/indicfairface_aligned",

    "img_size": 112,
    "embedding_dim": 512,
    "batch_size": 128,
    "epochs": 30,
    "freeze_backbone_epochs": 5,   # warm up the head, then unfreeze the backbone
    "lr_head": 1e-3,
    "lr_backbone": 1e-4,
    "weight_decay": 5e-4,
    "arc_s": 32.0,                 # ArcFace scale (32 is safe for small sets)
    "arc_m": 0.5,                  # ArcFace angular margin
    "num_workers": 4,
    "seed": 42,
    "out_dir": "outputs",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
os.makedirs(CFG["out_dir"], exist_ok=True)
print("device:", device,
      "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2 · Load EdgeFace-S and sanity-check its I/O

Official weights via `torch.hub`. The assert is deliberate: a wrong embedding
dim here is the #1 cause of silent downstream breakage.

In [ ]:
backbone = torch.hub.load(
    "otroshi/edgeface", "edgeface_s_gamma_05", source="github", pretrained=True
)
backbone = backbone.to(device)

backbone.eval()
with torch.no_grad():
    probe = backbone(torch.randn(2, 3, CFG["img_size"], CFG["img_size"], device=device))
print("embedding shape:", tuple(probe.shape))
assert probe.shape[1] == CFG["embedding_dim"], \
    f"expected {CFG['embedding_dim']}-d, got {probe.shape[1]}"
print("EdgeFace-S loaded OK.")

## 3 · Align faces → 112×112 (MTCNN)

**Critical:** ArcFace-family models need *tightly aligned* face crops (same as the
app's on-device crop). We detect + align with MTCNN and write a clean ImageFolder.
Run once per raw dataset; skip if `data_root` already exists.

**Dataset to use:** the **Indian Movie Face Database (IMFDB)** — 34,512 faces of
100 identities. Arrange the raw download as `data/indian_faces_raw/<actor>/*.jpg`.
Any identity-labelled set (CASIA, a custom field set) works the same way.

In [ ]:
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm.auto import tqdm

def align_dataset(raw_root, out_root, image_size=112, margin=0):
    if not os.path.isdir(raw_root):
        raise FileNotFoundError(
            f"{raw_root} not found. Put images as {raw_root}/<identity>/<img>.jpg"
        )
    # post_process=False -> face tensor in [0,255], so we save true pixels.
    mtcnn = MTCNN(image_size=image_size, margin=margin,
                  post_process=False, select_largest=True, device=device)
    identities = [d for d in sorted(os.listdir(raw_root))
                  if os.path.isdir(os.path.join(raw_root, d))]
    kept, dropped = 0, 0
    for identity in tqdm(identities, desc="aligning"):
        src_dir = os.path.join(raw_root, identity)
        dst_dir = os.path.join(out_root, identity)
        os.makedirs(dst_dir, exist_ok=True)
        for path in glob.glob(os.path.join(src_dir, "*")):
            try:
                img = Image.open(path).convert("RGB")
            except Exception:
                dropped += 1; continue
            face = mtcnn(img)
            if face is None:
                dropped += 1; continue
            arr = face.permute(1, 2, 0).clamp(0, 255).byte().cpu().numpy()
            Image.fromarray(arr).save(
                os.path.join(dst_dir, os.path.splitext(os.path.basename(path))[0] + ".jpg")
            )
            kept += 1
    print(f"aligned -> {out_root} | kept {kept}, dropped {dropped}")

# Run once (comment out after it has produced data_root):
if not os.path.isdir(CFG["data_root"]):
    align_dataset(CFG["data_root_raw"], CFG["data_root"], CFG["img_size"])
else:
    print("aligned dataset already present:", CFG["data_root"])

## 4 · Data loaders

Normalisation is **mean/std 0.5** — identical to the app so embeddings are
comparable. Light augmentation only (faces are already aligned).

In [ ]:
train_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.0),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(CFG["data_root"], transform=train_tf)
num_classes = len(train_ds.classes)
train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], drop_last=True, pin_memory=(device == "cuda"),
)
print(f"identities: {num_classes} | images: {len(train_ds)}")
assert num_classes >= 2, "need >= 2 identities to train ArcFace"


## 5 · ArcFace margin head

Standard additive angular margin (Deng et al., 2019). Adds margin `m` to the
target-class angle and scales by `s`, which pushes classes apart on the
hypersphere — the reason ArcFace embeddings verify so well.

In [ ]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=32.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_normal_(self.weight)
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        cosine = cosine.clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(1.0 - cosine ** 2)
        phi = cosine * self.cos_m - sine * self.sin_m
        # keep monotonic when theta + m > pi
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = one_hot * phi + (1.0 - one_hot) * cosine
        return output * self.s

## 6 · Assemble model + optimiser

We **freeze the backbone** for the first few epochs (let the fresh ArcFace head
settle), then **unfreeze** and fine-tune end-to-end at a smaller backbone LR so we
adapt without destroying the pretrained representation.

In [ ]:
head = ArcMarginProduct(CFG["embedding_dim"], num_classes,
                        s=CFG["arc_s"], m=CFG["arc_m"]).to(device)
criterion = nn.CrossEntropyLoss()

def set_backbone_trainable(flag: bool):
    for p in backbone.parameters():
        p.requires_grad = flag

set_backbone_trainable(False)  # warm-up: train head only
optimizer = torch.optim.AdamW(
    [
        {"params": head.parameters(), "lr": CFG["lr_head"]},
        {"params": backbone.parameters(), "lr": CFG["lr_backbone"]},
    ],
    weight_decay=CFG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
print("model + optimiser ready | classes:", num_classes)

## 7 · Verification-accuracy evaluation

Face recognition is judged by **verification** (same/different pairs), not
classification. We embed a folder, sample genuine/impostor pairs, and take the
best-threshold accuracy.

> **Honesty note:** measuring on the *training* folder overstates accuracy. For a
> real number, point `eval_folder` at held-out identities (`val_root`) that were
> **never trained on**. The training-set number is only a sanity signal.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, folder, transform, batch=128):
    ds = ImageFolder(folder, transform=transform)
    dl = DataLoader(ds, batch_size=batch, num_workers=CFG["num_workers"])
    model.eval()
    embs, labs = [], []
    for x, y in dl:
        e = F.normalize(model(x.to(device))).cpu()
        embs.append(e); labs.append(y)
    return torch.cat(embs).numpy(), torch.cat(labs).numpy()

def verification_accuracy(embs, labs, n_pairs=6000, seed=0):
    rng = np.random.default_rng(seed)
    by_label = {}
    for i, l in enumerate(labs):
        by_label.setdefault(int(l), []).append(i)
    usable = [l for l, v in by_label.items() if len(v) >= 2]
    if len(usable) < 2:
        return float("nan"), float("nan")
    pos, neg = [], []
    while len(pos) < n_pairs // 2:
        l = rng.choice(usable)
        a, b = rng.choice(by_label[l], size=2, replace=False)
        pos.append((a, b))
    labels_all = list(by_label.keys())
    while len(neg) < n_pairs // 2:
        l1, l2 = rng.choice(labels_all, size=2, replace=False)
        a = rng.choice(by_label[l1]); b = rng.choice(by_label[l2])
        neg.append((a, b))
    sims = np.array([float(embs[a] @ embs[b]) for a, b in pos] +
                    [float(embs[a] @ embs[b]) for a, b in neg])
    y = np.array([1] * len(pos) + [0] * len(neg))
    ths = np.linspace(-1, 1, 400)
    accs = [((sims > t) == y).mean() for t in ths]
    best = int(np.argmax(accs))
    return float(accs[best]), float(ths[best])

# Baseline (pretrained, before fine-tuning) for a before/after story:
_e, _l = extract_embeddings(backbone, CFG["data_root"], eval_tf)
base_acc, base_thr = verification_accuracy(_e, _l)
print(f"baseline verification acc (pretrained): {base_acc:.4f} @ cos>{base_thr:.3f}")

## 8 · Train

Saves the best backbone by verification accuracy.

In [ ]:
best_acc = 0.0
best_path = os.path.join(CFG["out_dir"], "edgeface_s_indian_best.pt")
eval_folder = CFG["val_root"] if os.path.isdir(CFG["val_root"]) else CFG["data_root"]

for epoch in range(CFG["epochs"]):
    if epoch == CFG["freeze_backbone_epochs"]:
        set_backbone_trainable(True)
        print(f"[epoch {epoch+1}] backbone unfrozen")

    backbone.train(); head.train()
    running = 0.0
    for x, y in tqdm(train_loader, desc=f"epoch {epoch+1}/{CFG['epochs']}"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            emb = backbone(x)
            logits = head(emb, y)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * x.size(0)
    scheduler.step()

    train_loss = running / len(train_ds)
    embs, labs = extract_embeddings(backbone, eval_folder, eval_tf)
    acc, thr = verification_accuracy(embs, labs)
    print(f"epoch {epoch+1}: loss={train_loss:.4f}  verif_acc={acc:.4f} @cos>{thr:.3f}")

    if acc == acc and acc > best_acc:   # acc==acc guards against NaN
        best_acc = acc
        torch.save(backbone.state_dict(), best_path)
        print(f"  ↳ saved best ({best_acc:.4f}) -> {best_path}")

print(f"done. best verification acc: {best_acc:.4f}  (baseline was {base_acc:.4f})")

## 9 · IndicFairFace bias evaluation (the slide number)

The headline "extra": show that fine-tuning **reduces geographic bias**. For each
state we compute the mean genuine (same-identity) cosine similarity, then report
the **variance across states** — lower = the model treats states more equally.
Compare pretrained vs fine-tuned.

Arrange IndicFairFace aligned as `data/indicfairface_aligned/<state>/<identity>/*.jpg`
(reuse `align_dataset` on each state folder). This cell is a **template** — adapt
the grouping to however you have the metadata.

In [ ]:
def per_group_genuine_variance(model, bias_root, transform):
    """mean genuine cosine per state -> variance across states (lower is fairer)."""
    if not os.path.isdir(bias_root):
        print("bias_root not found — skipping (optional):", bias_root)
        return None
    states = [d for d in sorted(os.listdir(bias_root))
              if os.path.isdir(os.path.join(bias_root, d))]
    means = {}
    for state in states:
        embs, labs = extract_embeddings(model, os.path.join(bias_root, state), transform)
        _, thr = verification_accuracy(embs, labs)  # not used, but validates pairs exist
        by = {}
        for i, l in enumerate(labs):
            by.setdefault(int(l), []).append(i)
        gen = []
        for idxs in by.values():
            for a in range(len(idxs)):
                for b in range(a + 1, len(idxs)):
                    gen.append(float(embs[idxs[a]] @ embs[idxs[b]]))
        if gen:
            means[state] = float(np.mean(gen))
    if len(means) < 2:
        print("need >= 2 states with genuine pairs"); return None
    var = float(np.var(list(means.values())))
    print("per-state mean genuine similarity:",
          {k: round(v, 3) for k, v in means.items()})
    print("inter-state variance:", round(var, 6))
    return var, means

# Fine-tuned model (best) vs pretrained — run both for the before/after number.
if os.path.isdir(CFG["bias_root"]):
    ft = torch.hub.load("otroshi/edgeface", "edgeface_s_gamma_05",
                        source="github", pretrained=True).to(device)
    ft.load_state_dict(torch.load(best_path, map_location=device))
    print("--- fine-tuned ---"); ft_var = per_group_genuine_variance(ft, CFG["bias_root"], eval_tf)
    print("--- pretrained ---"); pt_var = per_group_genuine_variance(backbone, CFG["bias_root"], eval_tf)

## 10 · Export → ONNX → **float16 TFLite**

We ship **float16** (not INT8) on purpose: the app feeds the recognition model
**normalised float** input (`config.ts` dtype `float32`, mean/std 0.5). A full-INT8
uint8-I/O model would need the app's preprocessing rewritten. Float16 is ~3.5 MB,
keeps float I/O, and drops in unchanged. (INT8 is offered at the end as optional.)

In [ ]:
# 10a · load best weights and export ONNX (NCHW, opset 13)
backbone.load_state_dict(torch.load(best_path, map_location=device))
backbone.eval()

onnx_path = os.path.join(CFG["out_dir"], "edgeface_s_indian.onnx")
dummy = torch.randn(1, 3, CFG["img_size"], CFG["img_size"], device=device)
torch.onnx.export(
    backbone, dummy, onnx_path,
    input_names=["input"], output_names=["embedding"],
    opset_version=13,
)
print("ONNX ->", onnx_path)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
out = sess.run(None, {"input": dummy.detach().cpu().numpy()})[0]
print("ONNX output shape:", out.shape)
assert out.shape[-1] == CFG["embedding_dim"]

In [ ]:
# 10b · ONNX (NCHW) -> TFLite (NHWC). onnx2tf emits float32/float16 variants and
# handles the layout transpose the app expects (112x112x3 HWC input).
tflite_dir = os.path.join(CFG["out_dir"], "tflite_edgeface")
!onnx2tf -i {onnx_path} -o {tflite_dir} -ois input:1,3,112,112

# Pick the float16 build and copy it to the app's expected filename.
import shutil, glob as _glob
cands = _glob.glob(os.path.join(tflite_dir, "*float16*.tflite")) or \
        _glob.glob(os.path.join(tflite_dir, "*float32*.tflite"))
assert cands, f"no tflite produced in {tflite_dir} — check the onnx2tf log above"
final_tflite = os.path.join(CFG["out_dir"], "edgeface_s.tflite")
shutil.copy(sorted(cands)[0], final_tflite)
print("TFLite ->", final_tflite, "| size(MB):", round(os.path.getsize(final_tflite)/1e6, 2))

## 11 · Verify the TFLite I/O and drop into the app

In [ ]:
import tensorflow as tf
interp = tf.lite.Interpreter(model_path=final_tflite)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
outp = interp.get_output_details()[0]
print("input :", inp["shape"], inp["dtype"])
print("output:", outp["shape"], outp["dtype"])

# Numerical parity check vs PyTorch on one sample.
x = np.random.rand(1, CFG["img_size"], CFG["img_size"], 3).astype(np.float32)
interp.set_tensor(inp["index"], x)
interp.invoke()
tfl = interp.get_tensor(outp["index"]).reshape(-1)
with torch.no_grad():
    pt = backbone(torch.tensor(x.transpose(0, 3, 1, 2), device=device)).cpu().numpy().reshape(-1)
cos = float(np.dot(tfl, pt) / (np.linalg.norm(tfl) * np.linalg.norm(pt) + 1e-9))
print("TFLite vs PyTorch cosine (want > 0.999):", round(cos, 5))
assert tuple(outp["shape"][-1:]) == (CFG["embedding_dim"],)

### Drop it into the app

1. Copy `outputs/edgeface_s.tflite` → `app/assets/models/edgeface_s.tflite`.
2. Register the asset in `app/src/face/modelAssets.ts`:
   ```ts
   export const RECOGNITION_ASSETS = {
     mobilefacenet: require('../../assets/models/mobilefacenet.tflite'),
     edgeface_s: require('../../assets/models/edgeface_s.tflite'),
   };
   ```
3. In `app/src/config.ts` set `ACTIVE_RECOGNITION = 'edgeface_s'` (the spec —
   112×112×3, mean/std 0.5, 512-d — already matches `RECOGNITION_MODELS.edgeface_s`).
4. **Open the file in Netron** and confirm input `1×112×112×3` float32 and output
   `1×512` — the app asserts the flattened input size at load, so a mismatch fails
   loudly rather than silently.
5. Re-enroll on device (the 512-d template differs from MobileFaceNet's 192-d, so
   old enrolments must be recreated).

### Optional · INT8 (only if you also change the app to feed uint8)
```python
# Full-INT8 needs a representative dataset AND app-side preprocessing changes
# (feed uint8 [0,255], let quantization scale). Left off by default on purpose.
# !onnx2tf -i {onnx_path} -o outputs/tflite_int8 -oiqt -cind input data/rep.npy [[0.5,0.5,0.5],[0.5,0.5,0.5]]
```
